# Metrics, Model Selection & Tuning

- [Evaluation Metrics](#Evaluation-Metrics)
  - [Confusion Matrix](#Confusion-Matrix)
  - [ROC-AUC, PR-AUC, threshold selection](#ROC-AUC,-PR-AUC,-threshold-selection)
  - [Log Loss, MCC, Cohen's Kappa](#Log-Loss,-MCC,-Cohen's-Kappa)
  - [Regression metrics — MAE, MSE, RMSE, MAPE, R²](#Regression-metrics-—-MAE,-MSE,-RMSE,-MAPE,-R²)
  - [Model calibration](#Model-calibration)
- [Model Selection and Cross-Validation](#Model-Selection-and-Cross-Validation)
  - [K-Fold Cross-Validation](#K-Fold-Cross-Validation)
  - [Stratified K-Fold](#Stratified-K-Fold)
  - [Nested Cross-Validation](#Nested-Cross-Validation)
  - [Learning curves](#Learning-curves)
  - [Grouped and time-series cross-validation](#Grouped-and-time-series-cross-validation)
- [Hyperparameter Tuning](#Hyperparameter-Tuning)
  - [Grid Search](#Grid-Search)
  - [Random Search](#Random-Search)
  - [Bayesian Optimization](#Bayesian-Optimization)

### Evaluation Metrics

#### Confusion Matrix

A 2x2 (binary) or KxK (multiclass) grid: rows = actual class, columns = predicted class.

Setup: spam classifier, 100 test emails — 20 actually spam, 80 actually not-spam. Model predicts 18 as spam, 82 as not-spam.

1. The four cells (binary case), "positive" = spam:
   - TP (True Positive): predicted spam, actually spam
   - TN (True Negative): predicted not-spam, actually not-spam
   - FP (False Positive) = Type I error: predicted spam, actually not-spam
   - FN (False Negative) = Type II error: predicted not-spam, actually spam

2. Worked numbers: of 20 actual spam, model catches 15 (TP=15, FN=5). Of 80 actual not-spam, model wrongly flags 3 (FP=3, TN=77).

               predicted spam   predicted not-spam
   actual spam        15 (TP)          5 (FN)
   actual not-spam     3 (FP)         77 (TN)

3. accuracy = (TP + TN) / (TP + TN + FP + FN) — fraction of all predictions that were correct, full stop.

4. Why it misleads under class imbalance: suppose instead only 2 of 100 emails are actually spam (heavy imbalance), and the model just predicts "not spam" for everything, every time.
   - TP=0, FN=2, FP=0, TN=98
   - accuracy = (0+98)/100 = 0.98 — looks great
   - but recall = 0/2 = 0 — it caught zero spam, the entire point of the classifier, while still scoring 98% accuracy

5. Multiclass extension: same idea, K×K instead of 2×2. Diagonal = correct. Off-diagonal cell [i,j] = actual class i predicted as class j — reading a specific off-diagonal cell tells you exactly which two classes get confused

6. Precision = TP/(TP+FP) — of everything predicted positive, how much really is
   Worked: 15/(15+3) = 15/18 ≈ 0.833

7. Recall = TP/(TP+FN) — of everything actually positive, how much did we catch
   Worked: 15/(15+5) = 15/20 = 0.75

8. F1 = 2·(P·R)/(P+R) — harmonic mean, penalizes a big P/R gap more than a plain average would
   Worked: 2·(0.833×0.75)/(0.833+0.75) = 2×0.625/1.583 ≈ 0.789

9. F-beta = (1+β²)·(P·R)/(β²·P+R) — generalizes F1; β>1 weights recall more, β<1 weights precision more. F1 is the special case β=1. Worked (β=2, recall-weighted — e.g. fraud, where catching cases matters more than false alarms): F2 = 5·(0.833×0.75)/(4×0.833+0.75) = 3.125/4.083 ≈ 0.766 (pulled lower than F1 toward recall=0.75, since β=2 cares less about precision's 0.833)

10. Specificity (TNR) = TN/(TN+FP) — of all actual negatives, how many correctly identified as negative (recall's mirror image, for the negative class) Worked (spam example): 77/(77+3) = 0.9625

New setup : for macro/micro/weighted — 3-class email routing, UNBALANCED on purpose (spam=5, promotions=5, primary=20 support, 30 total), so macro and weighted actually diverge this time:
    - spam:       TP=1, FN=4, FP=1 → precision=0.5, recall=0.2, F1≈0.286
    - promotions: TP=4, FN=1, FP=0 → precision=1.0, recall=0.8, F1≈0.889
    - primary:    TP=19, FN=1, FP=5 → precision≈0.792, recall=0.95, F1≈0.864

1. Macro avg F1 = (0.286+0.889+0.864)/3 ≈ 0.680 — treats spam's bad score (rare class, only 5 support) as equally important as primary's good one

2. Weighted avg F1 = (5×0.286 + 5×0.889 + 20×0.864)/30 ≈ 0.772 — pulled higher because primary (20/30 of the data) dominates the average, spam's ailure gets diluted to a fraction of its macro-avg impact

3. Micro avg: pool TP/FP/FN across ALL classes first, then compute one global precision/recall/F1, not an average of per-class scores.
    - ΣTP=1+4+19=24, ΣFP=1+0+5=6, ΣFN=4+1+1=6
    - micro_precision = 24/(24+6) = 0.8, micro_recall = 24/(24+6) = 0.8, micro_F1 = 0.8
    - Note: in single-label multiclass, ΣFP always equals ΣFN (every wrong prediction is simultaneously one FP for the predicted class and one FN for the true class) — s micro_precision = micro_recall = micro_F1 = accuracy exactly. Confirm: accuracy = 24/30 = 0.8

In [9]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score

import pandas as pd

# toy spam example: 1=spam, 0=not-spam
y_true = [1]*15 + [1]*5 + [0]*3 + [0]*77   # 20 actual spam (15 caught+5 missed), 80 actual not-spam (3 wrongly flagged+77 correct)
y_pred = [1]*15 + [0]*5 + [1]*3 + [0]*77   # matches the TP/FN/FP/TN split above

acc = accuracy_score(y_true, y_pred)
print("accuracy (spam example):", acc)

cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
cm_df = pd.DataFrame(cm, index=["actual spam", "actual not-spam"],
                      columns=["pred spam", "pred not-spam"])
print("confusion matrix (spam example):\n", cm_df)
# the misleading-imbalance case
y_true_imb = [1, 1] + [0]*98        # 2 actual spam, 98 actual not-spam
y_pred_imb = [0, 0] + [0]*98        # model predicts "not spam" for everything

acc_imb = accuracy_score(y_true_imb, y_pred_imb)
recall_imb = 0 / 2   # caught 0 of the 2 actual spam
print("\naccuracy (always predict not-spam):\n", acc_imb, "| recall:", recall_imb)


p = precision_score(y_true, y_pred)
r = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
f2 = fbeta_score(y_true, y_pred, beta=2)

print(f"precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}  f2={f2:.3f}")

# specificity for the binary spam example
spec = 77 / (77 + 3)
print("specificity:", spec)

# 3-class unbalanced example: 0=spam, 1=promotions, 2=primary
y_true_mc = [0]*5 + [1]*5 + [2]*20
y_pred_mc = [0,2,2,2,2] + [1,1,1,1,2] + [0]+[2]*19

print(classification_report(y_true_mc, y_pred_mc, target_names=["spam","promotions","primary"]))
print("accuracy:", accuracy_score(y_true_mc, y_pred_mc))

accuracy (spam example): 0.92
confusion matrix (spam example):
                  pred spam  pred not-spam
actual spam             15              5
actual not-spam          3             77

accuracy (always predict not-spam):
 0.98 | recall: 0.0
precision=0.833  recall=0.750  f1=0.789  f2=0.765
specificity: 0.9625
              precision    recall  f1-score   support

        spam       0.50      0.20      0.29         5
  promotions       1.00      0.80      0.89         5
     primary       0.79      0.95      0.86        20

    accuracy                           0.80        30
   macro avg       0.76      0.65      0.68        30
weighted avg       0.78      0.80      0.77        30

accuracy: 0.8


#### ROC-AUC, PR-AUC, threshold selection

Setup: 10 emails, true label (1=spam) and predicted spam probability, sorted by probability descending:
   - prob: 0.9,0.8,0.7,0.6,0.4,0.3,0.2,0.1,0.05,0.02
   - true: 1,  1,  0,  0,  1,  0,  0,  0,  0,   0     (3 actual spam, 7 not)

1. ROC = plot of TPR (recall) vs. FPR across every possible threshold.
   - TPR = TP/(TP+FN), FPR = FP/(FP+TN)
   - At threshold=0.75 (predict spam if prob≥0.75): only 0.9,0.8 flagged : TP=2, FP=0, FN=1, TN=7: TPR=2/3≈0.667, FPR=0/7=0
   - At threshold=0.5: 0.9,0.8,0.7,0.6 flagged → TP=2, FP=2, FN=1, TN=5 : TPR=2/3≈0.667, FPR=2/7≈0.286
   - At threshold=0.35: 0.9,0.8,0.7,0.6,0.4 flagged → TP=3, FP=2, FN=0, TN=5 : TPR=1.0, FPR=2/7≈0.286

   Lowering the threshold moves you along the curve: same TPR but worse FPR
   (0.75→0.5), then better TPR at that same FPR (0.5→0.35) — the actual
   curve is traced by sweeping every possible threshold, not just these 3.

2. AUC = area under that curve. Probabilistic interpretation: the chance a
   randomly picked actual-positive gets a higher predicted probability than
   a randomly picked actual-negative. AUC=1.0 = perfect ranking, AUC=0.5 =
   random guessing.

3. PR curve = precision vs. recall across the same thresholds (same TP/FP/FN,
   different ratio):
   threshold=0.75: P=2/2=1.0, R=2/3≈0.667
   threshold=0.5:  P=2/4=0.5, R=2/3≈0.667
   threshold=0.35: P=3/5=0.6, R=1.0
   Note precision *drops* as the threshold lowers and recall climbs — the
   classic precision/recall tradeoff, visible directly in the numbers.

4. ROC-AUC vs. PR-AUC: ROC-AUC can look deceptively good under heavy class
   imbalance (TN dominates FPR's denominator, so FPR stays low even with
   many false positives relative to the rare positive class). PR-AUC stays
   sensitive to this since it never involves TN at all — prefer PR-AUC when
   the positive class is rare (fraud, disease detection).

5. Threshold selection: not a math question, a business cost-asymmetry
   question. If a false positive (flagging a real email as spam) is costlier
   than a false negative (missing spam), pick a HIGH threshold (e.g. 0.75,
   precision=1.0 here). If missing a positive is costlier (e.g. missing
   fraud), pick a LOWER threshold to trade precision for recall.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc

y_true_prob = [1,1,0,0,1,0,0,0,0,0]
y_score = [0.9,0.8,0.7,0.6,0.4,0.3,0.2,0.1,0.05,0.02]

fpr, tpr, roc_thresholds = roc_curve(y_true_prob, y_score)
print("ROC-AUC:", roc_auc_score(y_true_prob, y_score))

precision, recall, pr_thresholds = precision_recall_curve(y_true_prob, y_score)
print("PR-AUC:", auc(recall, precision))

#### Log Loss, MCC, Cohen's Kappa

0. Log Loss (binary cross-entropy as an eval metric, not just a training loss):
   LogLoss = -(1/N) Σ [y·log(p) + (1-y)·log(1-p)]
   Using the same 10-email ROC/PR example (prob: 0.9,0.8,0.7,0.6,0.4,0.3,0.2,0.1,0.05,0.02;
   true: 1,1,0,0,1,0,0,0,0,0):
   - email 3 (true=0, p=0.7): -log(1-0.7) = -log(0.3) ≈ 1.204 — the single
     worst contributor, a confident WRONG prediction (said 70% spam, was not)
   - email 9 (true=0, p=0.05): -log(0.95) ≈ 0.051 — small penalty, confident
     and correct
   - sum across all 10 ≈ 4.120, LogLoss = 4.120/10 = 0.412
   Key property: log loss penalizes confident-and-wrong far more than
   unconfident-and-wrong — a hard 0/1 accuracy metric can't see this
   difference at all, only log loss (or any probability-based metric) can.

1. MCC (Matthews Correlation Coefficient), using the binary spam confusion
   matrix (TP=15, TN=77, FP=3, FN=5):
   MCC = (TP·TN − FP·FN) / √[(TP+FP)(TP+FN)(TN+FP)(TN+FN)]
   = (15×77 − 3×5) / √(18×20×80×82) = 1140 / √2,361,600 ≈ 1140/1536.75 ≈ 0.742
   Uses all four confusion matrix cells (including TN, which F1 ignores
   entirely) — more robust than F1 under imbalance since it can't be gamed
   by a model that's only ever evaluated on the positive class's neighborhood.

2. Cohen's Kappa — agreement beyond what chance alone would produce:
   κ = (p_o − p_e) / (1 − p_e)
   p_o = observed agreement = accuracy = 0.92 (spam example)
   p_e = expected agreement by chance = P(actual spam)·P(pred spam) +
         P(actual not-spam)·P(pred not-spam)
       = (20/100)(18/100) + (80/100)(82/100) = 0.036 + 0.656 = 0.692
   κ = (0.92 − 0.692) / (1 − 0.692) = 0.228/0.308 ≈ 0.740
   Close to MCC here (0.742) but not the same measure — kappa was designed
   for inter-rater agreement (two humans labeling the same data), MCC for
   classifier-vs-ground-truth; they often correlate on binary problems but
   diverge more on multiclass/imbalanced ones.

In [ ]:
from sklearn.metrics import log_loss, matthews_corrcoef, cohen_kappa_score

print("log loss:", log_loss(y_true_prob, y_score))
print("MCC:", matthews_corrcoef(y_true, y_pred))
print("Cohen's kappa:", cohen_kappa_score(y_true, y_pred))

#### Regression metrics — MAE, MSE, RMSE, MAPE, R²

0. Setup: predicting delivery time in minutes, 5 test cases.
   actual:    [30, 45, 20, 60, 35]
   predicted: [28, 50, 22, 55, 40]
   errors (actual-predicted): [2, -5, -2, 5, -5]

1. MAE = mean(|error|) — same units as the target, robust to outliers since
   errors aren't squared.
   |errors| = [2,5,2,5,5] → MAE = 19/5 = 3.8 (minutes)

2. MSE = mean(error²) — NOT same units as target (minutes²), penalizes large
   errors disproportionately (a 10-min miss contributes 4x a 5-min miss, not 2x).
   errors² = [4,25,4,25,25] → MSE = 83/5 = 16.6

3. RMSE = √MSE — back to the target's original units, still outlier-sensitive
   since the squaring happened before the sqrt.
   RMSE = √16.6 ≈ 4.074 (minutes)
   Notice RMSE (4.074) > MAE (3.8) whenever errors are uneven — RMSE is always
   ≥ MAE, and the gap grows with how much variance there is in error size.

4. MAPE = mean(|error|/|actual|) × 100 — scale-independent (a percentage),
   lets you compare error rates across totally different targets/units.
   |error|/|actual| = [2/30, 5/45, 2/20, 5/60, 5/35] ≈ [0.067,0.111,0.100,0.083,0.143]
   MAPE = mean(...) × 100 ≈ 10.08%
   Breaks down when actual values are near zero — division blows up, a small
   absolute error becomes a huge percentage.

5. R² (coefficient of determination) = 1 − SS_res/SS_tot — fraction of the
   target's variance the model explains.
   SS_res = Σerror² = 83 (sum, not mean, from step 2)
   mean(actual) = 190/5 = 38
   SS_tot = Σ(actual − mean)² = 64+49+324+484+9 = 930
   R² = 1 − 83/930 ≈ 0.911 — model explains ~91% of the variance in delivery time
   Can go negative: if the model is worse than just predicting the mean every
   time, SS_res > SS_tot.

6. Adjusted R² — penalizes adding predictors that don't actually help, since
   plain R² can only go up (never down) as you add more features, even
   useless ones.
   Formula: 1 − (1−R²)(n−1)/(n−p−1), n=samples, p=predictors
   With n=5, p=2: 1 − (1−0.911)(4/2) = 1 − 0.089×2 ≈ 0.822 — notably lower
   than plain R², reflecting the cost of using 2 predictors on only 5 samples.

7. Huber loss — hybrid of MAE/MSE: quadratic for small errors, linear for
   large ones, to stay outlier-robust while still being smooth/differentiable
   (MAE isn't differentiable at 0, which complicates gradient-based training).
   L(e) = 0.5e² if |e|≤δ, else δ(|e|−0.5δ)   (δ=3 here)
   e=2 (≤3): 0.5×4=2 | e=-5 (>3): 3×(5−1.5)=10.5 | e=-2: 2 | e=5: 10.5 | e=-5: 10.5
   mean Huber = 35.5/5 = 7.1 — notice large errors (magnitude 5) contribute
   only 10.5 each here vs. 25 each under plain MSE — Huber caps the outlier penalty.

8. Residual analysis — not a single number, a plot (residuals vs. predicted
   values). Look for: residuals scattered randomly around 0 with constant
   spread (homoscedasticity, good) vs. a funnel/curve shape (heteroscedasticity
   or a missing nonlinear relationship — the model is systematically wrong in
   some region, which no single aggregate metric like RMSE would reveal).

9. Choosing MAE vs. RMSE vs. MAPE: RMSE when large errors are especially
   costly (penalize them harder); MAE when every unit of error should count
   equally and outliers exist that shouldn't dominate the metric; MAPE when
   comparing error rates across different scales/units — but never when
   actual values can be near zero.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

actual = np.array([30, 45, 20, 60, 35])
predicted = np.array([28, 50, 22, 55, 40])

mae = mean_absolute_error(actual, predicted)
mse = mean_squared_error(actual, predicted)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((actual - predicted) / actual)) * 100
r2 = r2_score(actual, predicted)

print(f"MAE={mae:.3f}  MSE={mse:.3f}  RMSE={rmse:.3f}  MAPE={mape:.2f}%  R2={r2:.3f}")

# adjusted R2, n=5 samples, p=2 predictors (illustrative)
n, p = 5, 2
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R2:", adj_r2)

# Huber loss, delta=3
delta = 3
errors = actual - predicted
huber = np.where(np.abs(errors) <= delta, 0.5 * errors**2, delta * (np.abs(errors) - 0.5 * delta))
print("Huber loss per point:", huber, "| mean:", huber.mean())

# residual plot
residuals = actual - predicted
plt.scatter(predicted, residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("predicted")
plt.ylabel("residual (actual - predicted)")
plt.title("Residual plot")
plt.show()

#### Model calibration

A predicted probability being useful as a PROBABILITY, not just as a ranking signal, is a separate property from accuracy. A model can correctly rank examples (high scores for positives, low for negatives, good AUC) while still being badly calibrated, its 0.8 predictions might only be correct 55% of the time, not 80%.

0. Reliability diagram, the diagnostic: bucket predictions by confidence, compare each bucket's AVERAGE predicted probability to the OBSERVED frequency of the positive class within that bucket.

Worked example: 10 predictions land in the [0.7, 0.8) confidence bucket, average predicted probability in that bucket = 0.75. Of those 10, only 6 were actually the positive class, observed frequency = 0.6. The model is overconfident in this bucket, it says "75% likely" when reality is closer to 60%. A perfectly calibrated model's reliability diagram is the diagonal line, predicted probability equals observed frequency at every bucket.

1. Platt scaling: fit a simple logistic regression on top of the model's raw output scores (not the original features), mapping raw score -> calibrated probability. Cheap, works well when the miscalibration itself has a roughly sigmoid shape (common for SVM margins, and for some boosted tree score distributions).

2. Isotonic regression: fits a non-parametric, monotonic step function instead of assuming a sigmoid shape, more flexible, can correct more complex miscalibration patterns. Needs more calibration data to avoid overfitting the step function to noise, since it has many more effective degrees of freedom than Platt's single logistic curve.

3. Why this matters beyond the diagram: any decision that uses the predicted probability directly, not just the predicted class, depends on calibration. A fraud system deciding "auto-block above 0.9, route to review between 0.5 and 0.9" is trusting the model's probabilities to mean what they say, a miscalibrated 0.9 that is really only correct 70% of the time would auto-block far more false positives than the threshold was designed to allow.

In [ ]:
import numpy as np
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=300, n_features=5, random_state=42)
X_train, y_train = X[:200], y[:200]
X_test, y_test = X[200:], y[200:]

# SVM decision scores are notoriously poorly calibrated as raw probabilities
raw_model = SVC(probability=True, random_state=42).fit(X_train, y_train)
raw_probs = raw_model.predict_proba(X_test)[:, 1]

platt_model = CalibratedClassifierCV(SVC(random_state=42), method="sigmoid", cv=3).fit(X_train, y_train)
platt_probs = platt_model.predict_proba(X_test)[:, 1]

isotonic_model = CalibratedClassifierCV(SVC(random_state=42), method="isotonic", cv=3).fit(X_train, y_train)
isotonic_probs = isotonic_model.predict_proba(X_test)[:, 1]

for name, probs in [("raw", raw_probs), ("Platt", platt_probs), ("isotonic", isotonic_probs)]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=5)
    print(f"{name}: mean predicted per bin={mean_pred.round(2)}, observed frequency per bin={frac_pos.round(2)}")

### Model Selection and Cross-Validation

Used informally throughout this series ("held out on val", "stratify=df['typology']" in the fraud project) without ever explaining the mechanics as their own topic. Covers k-fold, stratified k-fold, nested CV.

#### K-Fold Cross-Validation

A single train/val split gives one noisy estimate of generalization performance, it depends heavily on which rows happened to land in val. K-fold CV: split the data into k equal folds, for each fold, train on the other k-1 folds and validate on the held-out one, average the k validation scores. Every row gets used for validation exactly once, and for training k-1 times.

Toy setup, 10 examples, k=5 (2 examples per fold):
```
fold0=[0,1]  fold1=[2,3]  fold2=[4,5]  fold3=[6,7]  fold4=[8,9]

split 1: train on folds 1-4, validate on fold 0
split 2: train on folds 0,2,3,4, validate on fold 1
... (5 splits total, one per fold as the held-out validation set)
```
Final estimate: average of the 5 validation scores, plus the standard deviation across them, which itself is useful information, a high spread across folds means the model's performance is unstable/sensitive to which rows it happened to see, a low spread means the estimate is reliable.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
X = rng.normal(size=(10, 3))
y = (X[:, 0] > 0).astype(int)

kf = KFold(n_splits=5)
for fold_num, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"fold {fold_num}: train_idx={train_idx}, val_idx={val_idx}")

scores = cross_val_score(LogisticRegression(), X, y, cv=5)
print("\nfold scores:", scores.round(3))
print("mean:", scores.mean().round(3), "| std:", scores.std().round(3))

#### Stratified K-Fold

**The problem plain k-fold has with imbalanced classes.**

Worked example, 10 examples, labels [0,0,0,0,0,0,0,0,1,1] (imbalanced, only 2 of class 1, at indices 8 and 9). Plain k-fold, splitting sequentially into 5 folds of 2:
```
fold0=[0,1] fold1=[2,3] fold2=[4,5] fold3=[6,7] fold4=[8,9]
```
fold4 contains BOTH class-1 examples, and only them. Consequences:
- When fold4 is the validation set: the training data (folds 0-3) has ZERO class-1 examples. The model never sees class 1 during training for this split, cannot possibly predict it.
- When any of folds 0-3 is the validation set: that validation set has ZERO class-1 examples. Cannot evaluate class-1 recall/precision at all in those 4 of the 5 splits.

Stratified k-fold fixes this by ensuring each fold gets a proportional share of each class, the 2 class-1 examples get spread across different folds instead of landing together by chance, e.g. one in fold0, one in fold2, none of the folds end up with zero class-1 representation on both sides of the split. Directly relevant to the fraud-theme-detection project's own train_test_split call, which used `stratify=df['typology']` for exactly this reason, keeping the 200-per-theme balance intact across train/val/test.

In [ ]:
from sklearn.model_selection import StratifiedKFold

y_imbalanced = np.array([0,0,0,0,0,0,0,0,1,1])
X_imbalanced = np.arange(10).reshape(-1, 1)

print("plain KFold:")
for train_idx, val_idx in KFold(n_splits=5).split(X_imbalanced):
    val_labels = y_imbalanced[val_idx]
    print(f"  val_idx={val_idx}, val labels={val_labels}")

print("\nStratifiedKFold:")
for train_idx, val_idx in StratifiedKFold(n_splits=5).split(X_imbalanced, y_imbalanced):
    val_labels = y_imbalanced[val_idx]
    print(f"  val_idx={val_idx}, val labels={val_labels}")

#### Nested Cross-Validation

**The problem plain CV has when you are ALSO tuning hyperparameters.**

If you use k-fold CV to pick the best hyperparameters (as in the Hyperparameter Tuning section's GridSearchCV/RandomizedSearchCV), and then report THAT SAME cross-validated score as your estimate of how well the model generalizes, the estimate is optimistic. The hyperparameter search already saw and optimized against those exact validation folds, the reported score reflects some amount of fitting to the validation folds themselves, not genuinely unseen data, the same leakage principle as evaluating on data the model was tuned against, one level removed from training data itself but still not fully honest.

**The fix: two nested loops.**

Outer loop: k-fold split for the FINAL performance estimate. Inner loop: within each outer training fold, run a full hyperparameter search (its own k-fold CV) to pick the best config, then evaluate that config on the outer loop's held-out fold, which the inner search never touched.
```
for outer_train, outer_test in outer_kfold.split(X):
    best_params = grid_search(outer_train)     # inner CV, tunes hyperparameters
    score = evaluate(best_params, outer_test)  # outer_test was NEVER used by the inner search
    outer_scores.append(score)

final estimate = mean(outer_scores)
```
Expensive, k_outer * k_inner total model fits instead of just k, but it is the honest way to report generalization performance when hyperparameter tuning is part of the pipeline being evaluated, not an afterthought bolted onto a single CV run.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

param_grid = {"C": [0.1, 1.0, 10.0]}

# nested CV: GridSearchCV (inner loop) wrapped inside cross_val_score (outer loop)
inner_search = GridSearchCV(LogisticRegression(), param_grid, cv=3)
nested_scores = cross_val_score(inner_search, X, y, cv=3)

print("nested CV scores (honest generalization estimate):", nested_scores.round(3))
print("mean:", nested_scores.mean().round(3))

#### Learning curves

Learning curve: train/val score vs training-set size. High bias: both low and converge close → more capacity/features. High variance: large train–val gap → more data/regularization. Validation curve: score vs one hyperparameter (e.g. depth) to find the sweet spot.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve
X, y = make_classification(3000, 20, n_informative=5, random_state=0)
sizes, train, val = learning_curve(LogisticRegression(max_iter=1000), X, y, train_sizes=[0.05, 0.1, 0.25, 0.5, 1.0], cv=5)
for s, a, b in zip(sizes, train.mean(1), val.mean(1)):
    print(f"train size {s:5d} | train accuracy {a:.3f} | validation accuracy {b:.3f}")

train size   120 | train accuracy 0.883 | validation accuracy 0.841
train size   240 | train accuracy 0.917 | validation accuracy 0.867
train size   600 | train accuracy 0.893 | validation accuracy 0.870
train size  1200 | train accuracy 0.883 | validation accuracy 0.870
train size  2400 | train accuracy 0.880 | validation accuracy 0.877


#### Grouped and time-series cross-validation

k-fold: average over k splits. Stratified: preserves class ratio (imbalance). Grouped: keep the same user/entity in one fold (no leakage). Time-series split: expanding/rolling window, train before validation. Nested CV: inner loop tunes hyperparameters, outer loop gives an unbiased estimate.

The label here is random per customer, so no model can truly predict it. Random k-fold still scores well above chance because the same customer appears in train and validation, while grouped k-fold shows chance-level performance, which is the honest answer.

In [2]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
rng = np.random.default_rng(0)
n_cust, per = 300, 5
cust = np.repeat(np.arange(n_cust), per)
signature = rng.normal(size=n_cust)[cust]                         # a feature that identifies the customer
X = np.c_[signature + rng.normal(0, 0.001, cust.size), rng.normal(size=cust.size)]
y = np.repeat(rng.random(n_cust) < 0.5, per).astype(int)          # label is a random property of the customer
rf = RandomForestClassifier(n_estimators=100, random_state=0)
print("random KFold AUC:", cross_val_score(rf, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean().round(3))
print("GroupKFold   AUC:", cross_val_score(rf, X, y, cv=GroupKFold(5), groups=cust, scoring="roc_auc").mean().round(3))

random KFold AUC: 0.769


GroupKFold   AUC: 0.56


### Hyperparameter Tuning

Referenced constantly throughout this series ("tune n_estimators", "lower learning_rate") without covering how the tuning itself is actually done. Covers Grid Search, Random Search, Bayesian Optimization.

#### Grid Search

Define a fixed grid of candidate values for each hyperparameter, exhaustively try every combination, keep whichever combination scored best on a held-out validation set (or cross-validation).

Worked example: n_estimators in {50, 100, 200}, max_depth in {3, 5, 7}. That is 3x3=9 combinations to train and evaluate, every one of them, no shortcuts.
```
(50,3) (50,5) (50,7)
(100,3) (100,5) (100,7)
(200,3) (200,5) (200,7)
```
Combinatorial explosion: add a third hyperparameter with 3 candidate values, learning_rate in {0.01, 0.1, 0.3}, and the grid jumps to 3x3x3=27 combinations. 5 hyperparameters at 3 values each is 3^5=243. Grid search's cost grows exponentially in the number of tuned hyperparameters, this is its core practical limitation.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

rng = np.random.default_rng(0)
X = rng.normal(size=(100, 5))
y = (X[:, 0] + X[:, 1] > 0).astype(int)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)
grid.fit(X, y)

print("combinations tried:", len(grid.cv_results_["params"]))
print("best params:", grid.best_params_)
print("best score:", grid.best_score_)

#### Random Search

Instead of an exhaustive grid, sample a fixed NUMBER of random combinations from the search space (a distribution per hyperparameter, not just a discrete list). Counterintuitively often more efficient than grid search, even though it looks less thorough.

Worked argument: same 2 hyperparameters, but suppose A (say learning_rate) genuinely matters a lot for performance, and B (say a minor regularization knob) barely matters at all. With grid search's 3x3=9 combinations, only 3 DISTINCT values of A ever get tested, the grid ties A and B's resolution together, wasting 6 of the 9 evaluations on redundant A-values paired with different B's.
```
grid search unique A-values tested: 3   (each tested 3 times, with different B)
random search, 9 random draws: each draw picks A independently from a continuous
  range, with high probability close to 9 distinct A-values get tested
```
Since A is the hyperparameter that actually matters, random search's 9 draws explore that important dimension roughly 3x more thoroughly than grid search's 9 evaluations did, for the identical evaluation budget. This is the actual finding from Bergstra and Bengio's original random search paper: when only a few hyperparameters matter (nearly always true in practice), random search finds equally good or better configurations than grid search using the same compute budget.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    "n_estimators": randint(50, 300),
    "max_depth": randint(2, 10),
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, n_iter=9, cv=3, random_state=42)
random_search.fit(X, y)

unique_depths_tried = set(p["max_depth"] for p in random_search.cv_results_["params"])
print("unique max_depth values tried:", len(unique_depths_tried), "of 9 total evaluations")
print("best params:", random_search.best_params_)
print("best score:", random_search.best_score_)

#### Bayesian Optimization

Grid and random search both ignore what has already been learned, every evaluation is independent, picked without regard to previous results. Bayesian optimization builds a probabilistic model (commonly a Gaussian Process) of "hyperparameters -> validation score" AS IT GOES, using that model to decide where to sample next, balancing exploitation (try values near the best-so-far) against exploration (try values in still-uncertain regions of the space).

**Why this is more sample-efficient.**

Grid and random search waste evaluations on clearly bad regions of the space that could have been ruled out after just a few samples. Bayesian optimization's surrogate model learns the rough shape of the objective function (which regions tend to score well) after only a handful of evaluations, and concentrates the remaining budget where it is likely to pay off, this matters most when each evaluation is expensive (training a large boosted model can take minutes to hours), a regime where grid/random search's wasted evaluations are genuinely costly, not just slightly inefficient.

**Practical notes.**

Diminishing returns advantage when evaluations are cheap and the search space is small, the overhead of fitting the surrogate model is not worth it, random search alone is usually fine there. Most valuable specifically for expensive-to-train models (XGBoost/LightGBM/CatBoost with many trees, or any neural network) and large hyperparameter spaces. Optuna and scikit-optimize are the common libraries, `pip install optuna`, not installed in this environment.

In [ ]:
# illustrative only, optuna is not installed in this environment
# import optuna
#
# def objective(trial):
#     n_estimators = trial.suggest_int("n_estimators", 50, 300)
#     max_depth = trial.suggest_int("max_depth", 2, 10)
#     model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
#     from sklearn.model_selection import cross_val_score
#     return cross_val_score(model, X, y, cv=3).mean()
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)
# print("best params:", study.best_params)
print("see commented block above, requires: pip install optuna")